In [1]:
from azureml.core import Workspace, Dataset, Datastore

# اتصال به Workspace
subscription_id = 'f8c5aac3-29fc-4387-858a-1f61722fb57a'
resource_group = 'forskerpl-n0ybkr-rg'
workspace_name = 'forskerpl-n0ybkr-mlw'

ws = Workspace(subscription_id=subscription_id,
               resource_group=resource_group,
               workspace_name=workspace_name)

# گرفتن datastore
datastore = Datastore.get(ws, "researcher_data")

# خواندن همه فایل‌های parquet در مسیر مشخص
dataset = Dataset.Tabular.from_parquet_files(
    path=[(datastore, 'Zahra/012026/Data/MEDS_MDP/data/held_out/*.parquet')]    #     Zahra/Data-07-2025/MDP/MEDS_811/data/train
)

# تبدیل به pandas DataFrame
df = dataset.to_pandas_dataframe()
df.head(15)


/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.1.1) and mlflow-skinny (2.22.1) are different. This may lead to unexpected behavior. Please install the same version of both packages.
  mlflow.mismatch._check_version_mismatch()


Resolving access token for scope "https://storage.azure.com/.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}


,subject_id,time,code,numeric_value
0,5,1992-03-12 00:00:00,DOB,NaN
1,5,2017-04-16 00:00:00,D/DR040,NaN
2,5,2017-04-16 21:43:00,ADMISSION_ADT,NaN
3,5,2017-04-16 21:43:00,MOVE_ADT,NaN
4,5,2017-04-16 21:43:00,^AFSNIT_ADT/,NaN
5,5,2017-04-16 23:07:00,ADMISSION_ADT,NaN
6,5,2017-04-16 23:07:00,MOVE_ADT,NaN
7,5,2017-04-16 23:07:00,^AFSNIT_ADT/,NaN
8,5,2017-04-16 23:10:00,P/AAF3,NaN
9,5,2017-04-17 07:35:00,ADMISSION_ADT,NaN


In [2]:
len(df)

55907759

In [3]:
import pandas as pd

# فرض بر این که فایل CSV رو داری
# df = pd.read_csv("your_file.csv")  # یا مستقیم اگر DataFrame آماده‌ست، نیازی نیست

# دسته‌بندی هر subject_id بر اساس وجود کدهای مختلف
m_patients = df[df['code'].str.startswith('M/', na=False)]['subject_id'].unique()
p_patients = df[df['code'].str.startswith('P/', na=False)]['subject_id'].unique()
d_patients = df[df['code'].str.startswith('D/', na=False)]['subject_id'].unique()
s_patients = df[df['code'].str.startswith('S/', na=False)]['subject_id'].unique()

# کل بیماران منحصربه‌فرد
all_patients = df['subject_id'].unique()

# نمایش آمار
print(f"Whole Patients: {len(all_patients)}")
print(f"The patients has M-medication Codes: {len(m_patients)}")
print(f"The patients has D-diagnosis Codes: {len(d_patients)}")
print(f"The patients has P-Procedure Codes: {len(p_patients)}")
print(f"The patients has S-SKS Codes: {len(s_patients)}")


Whole Patients: 221803
The patients has M-medication Codes: 151356
The patients has D-diagnosis Codes: 221749
The patients has P-Procedure Codes: 212660
The patients has S-SKS Codes: 0


In [4]:
subject_counts = df['subject_id'].value_counts()


In [5]:
subject_counts

106        63789
1154761    48106
1011632    45271
1580453    44344
926005     43380
           ...  
392598         3
1351173        3
85775          2
1682596        2
1506679        2
Name: subject_id, Length: 221803, dtype: int64

In [6]:
p_Num = df[df['code'].str.startswith('P/', na=False)]

In [7]:
p_Num

,subject_id,time,code,numeric_value
8,5,2017-04-16 23:10:00,P/AAF3,NaN
16,5,2017-08-09 15:37:00,P/BWAA21,NaN
17,5,2017-08-09 15:39:00,P/AAF3,NaN
31,5,2017-12-14 08:30:00,P/UXUC80,NaN
34,5,2017-12-14 15:08:00,P/UXRC00,NaN
...,...,...,...,...
55907749,2217869,2024-04-03 10:41:00,P/BWHB40A,NaN
55907750,2217869,2024-04-03 10:41:00,P/BWHB82,NaN
55907753,2217869,2024-05-08 09:47:00,P/BBHF32,NaN
55907754,2217869,2024-05-08 09:47:00,P/BWHB40A,NaN


In [8]:
import pandas as pd

# پیدا کردن سطرهایی که فقط codeهای نوع /P دارن
only_p = df[df['code'].str.startswith('P/', na=False)]

# بیماران با فقط /P کد
subject_ids_only_p = only_p['subject_id'].unique()

# حالا بیماران با codeهای غیر از /P
not_p = df[~df['code'].str.startswith('P/', na=False)]
subject_ids_with_non_p = set(not_p['subject_id'].unique())

# حذف بیمارانی که فقط /P دارن
only_p_ids_to_exclude = [sid for sid in subject_ids_only_p if sid not in subject_ids_with_non_p]

print("Number of patients with only Surgery code: ", only_p_ids_to_exclude)


Number of patients with only Surgery code:  []


In [9]:
df_filtered = df[~df['code'].str.startswith('P/', na=False)]

In [10]:
subject_counts_MDS = df_filtered['subject_id'].value_counts()

In [11]:
subject_counts_MDS

106        60295
1154761    47724
1580453    43789
1011632    43454
926005     40467
           ...  
1629822        2
528769         2
1044530        2
190688         2
1335726        2
Name: subject_id, Length: 221803, dtype: int64

In [12]:
subject_counts_df = subject_counts.reset_index()
subject_counts_df.columns = ['subject_id', 'original_count']

subject_counts_MDS_df = subject_counts_MDS.reset_index()
subject_counts_MDS_df.columns = ['subject_id', 'new_count']


In [13]:
import pandas as pd
comparison_df = pd.merge(subject_counts_df, subject_counts_MDS_df, on='subject_id', how='outer')


In [14]:
comparison_df['difference'] =  comparison_df['original_count'] - comparison_df['new_count']


In [15]:
comparison_df = comparison_df.sort_values(by='difference', ascending=False)


In [16]:
comparison_df

,subject_id,original_count,new_count,difference
0,106,63789,60295,3494
4,926005,43380,40467,2913
22,1348315,21168,18439,2729
17,1665595,24140,21481,2659
59,1063572,13132,10666,2466
...,...,...,...,...
199696,1068494,14,14,0
166024,2125938,28,28,0
166019,1491223,28,28,0
166016,1999951,28,28,0


In [17]:
unchanged_count = (comparison_df['difference'] == 0).sum()
print("unchanged_count", unchanged_count)


unchanged_count 9143


In [18]:
comparison_df['abs_diff'] = comparison_df['difference'].abs()
most_changed = comparison_df.sort_values(by='abs_diff', ascending=False)


In [19]:
print(most_changed.head(10))


     subject_id  original_count  new_count  difference  abs_diff
0           106           63789      60295        3494      3494
4        926005           43380      40467        2913      2913
22      1348315           21168      18439        2729      2729
17      1665595           24140      21481        2659      2659
59      1063572           13132      10666        2466      2466
23      1708135           20637      18306        2331      2331
27      1759975           19223      17036        2187      2187
200      965519            8226       6261        1965      1965
482     1536302            5826       3888        1938      1938
44      1183092           14920      13043        1877      1877


In [20]:
changed_df = comparison_df[comparison_df['difference'] != 0]
min_new_count = changed_df['new_count'].min()
lowest_new_count_patients = changed_df[changed_df['new_count'] == min_new_count]


In [21]:
lowest_new_count_patients = lowest_new_count_patients.rename(
    columns={
        'original_count': 'MDP codes',
        'new_count': 'MD codes'
    }
)


In [22]:
lowest_new_count_patients

,subject_id,MDP codes,MD codes,difference,abs_diff
179993,816167,21,2,19,19
189609,1755398,17,2,15,15
199370,369026,14,2,12,12
205784,1470335,12,2,10,10
207229,528769,11,2,9,9
209546,668277,10,2,8,8
208749,542733,10,2,8,8
210397,11837,9,2,7,7
213488,1044530,7,2,5,5
213374,375144,7,2,5,5


.str.upper() شرط را case-insensitive می‌کند

بل از فیلتر، ستون را به pd.StringDtype() تبدیل می‌کند؛ این کار رفتار .str را پایدار و قابل پیش‌بینی می‌کند (<NA> به‌جای NaN)



In [23]:
# کل بیماران منحصربه‌فرد
all_patients = df_filtered['subject_id'].unique()
len(all_patients)
# نمایش آمار

221803

In [24]:
len(df_filtered)

45391200

In [25]:
import pandas as pd
import numpy as np

# امن‌تر: اگر code نال یا غیررشته‌ای بود اذیت نکنه
df['code'] = df['code'].astype('string')
df_filtered = df[~df['code'].str.upper().str.startswith('P/', na=False)].copy()
print("kept rows MD:", len(df_filtered), " / total:", len(df))


In [ ]:
import pandas as pd
import numpy as np

# اطمینان از نوع‌ها (برای خروجی تمیز و بدون خطا)
df_filtered = df_filtered.copy()
df_filtered['subject_id']    = pd.to_numeric(df_filtered['subject_id'], errors='coerce').astype('Int64')
df_filtered['numeric_value'] = pd.to_numeric(df_filtered['numeric_value'], errors='coerce').astype('float32')
df_filtered['time']          = pd.to_datetime(df_filtered['time'], errors='coerce', utc=False)

# ردیف‌های بدون subject_id را حذف کنیم (نمی‌توان شارد کرد)
df_filtered = df_filtered.dropna(subset=['subject_id']).copy()
df_filtered['subject_id'] = df_filtered['subject_id'].astype('int64')

# ۳۶ شارد: هر بیمار فقط در یک فایل (mod 36)
#N_SHARDS = 45
#df_filtered['__shard__'] = (df_filtered['subject_id'] % N_SHARDS).astype('int16')

print("rows to write:", len(df_filtered))


In [ ]:
import numpy as np
import os

N_SHARDS = 5   #45 for Whole # 36 when we have split
OUT_DIR = "./_held_outMDP_withoutP_sharded"
os.makedirs(OUT_DIR, exist_ok=True)

cols_out = ['subject_id', 'time', 'code', 'numeric_value']

# فرض: subject_id قبلاً int64 شده و NaNها حذف شده‌اند (طبق سلول قبلی‌ات)
sid_mod = (df_filtered['subject_id'].to_numpy(dtype=np.int64, copy=False) % N_SHARDS)

written = 0
for k in range(N_SHARDS):
    mask = (sid_mod == k)                 # بدون ستون اضافی، فقط یک آرایهٔ NumPy
    part = df_filtered.loc[mask, cols_out].sort_values(['subject_id','time'])
    # اگر می‌خوای حتماً ۳۶ فایل 0..35 داشته باشی حتی اگه خالی باشن:
    # if part.empty:
    #     part = part.iloc[0:0]  # فایل صفر-سطر با همان ستون‌ها
    part.to_parquet(os.path.join(OUT_DIR, f"{k}.parquet"),
                    engine="pyarrow", compression="snappy", index=False)
    written += 1

print(f"Done. wrote {written} parquet files into {OUT_DIR}")


In [ ]:
import pyarrow.parquet as pq

OUT_DIR = "./_held_outMDP_withoutP_sharded"

def parquet_num_rows(path):
    pf = pq.ParquetFile(path)
    md = pf.metadata
    return sum(md.row_group(i).num_rows for i in range(md.num_row_groups))

total = 0
for f in sorted(p for p in os.listdir(OUT_DIR) if p.endswith('.parquet')):
    n = parquet_num_rows(os.path.join(OUT_DIR, f))
    total += n
    print(f, "rows:", n)
print("TOTAL rows:", total)


In [ ]:
from azureml.data.datapath import DataPath
from azureml.data.dataset_factory import FileDatasetFactory
import os

OUT_DIR = "./_held_outMDP_withoutP_sharded"
DST_PREFIX = "Zahra/012026/Data/MEDS_MD/data/held_out"  #held_out"  # بدون اسلشِ اول
''

# اطمینان: پوشه خروجی وجود دارد و فایل parquet داخلش هست
print("Local files to upload:", len([f for f in os.listdir(OUT_DIR) if f.endswith(".parquet")]))

# مقصد روی Datastore
target = DataPath(datastore, DST_PREFIX)

# آپلود همه محتویات OUT_DIR به DST_PREFIX
_ = FileDatasetFactory.upload_directory(
    src_dir=OUT_DIR,
    target=target,
    overwrite=True,
    show_progress=True,
)
print(f"Uploaded to datastore path: {DST_PREFIX}")


In [ ]:
paths = Dataset.File.from_files(path=[(datastore, f"{DST_PREFIX}/*.parquet")]).to_path()
print("Found in datastore:", len(paths))
print(paths[:20])
